In [1]:
import pyspark
import os
from pyspark.sql import SparkSession
from pyspark.sql.types import *
from pyspark.sql.functions import *
filepath="D:/bda_pyspark/datasets/indian_food.csv"

In [2]:
def createSparkSession():
    spark=SparkSession.builder.appName('indian-food').getOrCreate()
    return spark

In [3]:
import pandas as pd
df=pd.read_csv(filepath)
df

,name,ingredients,diet,prep_time,cook_time,flavor_profile,course,state
0,Balu shahi,"Maida flour, yogurt, oil, sugar",vegetarian,45,25,sweet,dessert,West Bengal
1,Boondi,"Gram flour, ghee, sugar",vegetarian,80,30,sweet,dessert,Rajasthan
2,Gajar ka halwa,"Carrots, milk, sugar, ghee, cashews, raisins",vegetarian,15,60,sweet,dessert,Punjab
3,Ghevar,"Flour, ghee, kewra, milk, clarified butter, su...",vegetarian,15,30,sweet,dessert,Rajasthan
4,Gulab jamun,"Milk powder, plain flour, baking powder, ghee,...",vegetarian,15,40,sweet,dessert,West Bengal
...,...,...,...,...,...,...,...,...
250,Til Pitha,"Glutinous rice, black sesame seeds, gur",vegetarian,5,30,sweet,dessert,Assam
251,Bebinca,"Coconut milk, egg yolks, clarified butter, all...",vegetarian,20,60,sweet,dessert,Goa
252,Shufta,"Cottage cheese, dry dates, dried rose petals, ...",vegetarian,-1,-1,sweet,dessert,Jammu & Kashmir
253,Mawa Bati,"Milk powder, dry fruits, arrowroot powder, all...",vegetarian,20,45,sweet,dessert,Madhya Pradesh


In [4]:
def create_dataFrame(spark,filepath):
    df=spark.read.csv(filepath,header=True,inferSchema=True)
    return df

In [5]:
spark=createSparkSession()
df=create_dataFrame(spark,filepath)
df.printSchema()

root
 |-- name: string (nullable = true)
 |-- ingredients: string (nullable = true)
 |-- diet: string (nullable = true)
 |-- prep_time: integer (nullable = true)
 |-- cook_time: integer (nullable = true)
 |-- flavor_profile: string (nullable = true)
 |-- course: string (nullable = true)
 |-- state: string (nullable = true)



In [12]:
def clean_dataset(df):
    df=df.drop("ingredients")

    for c in df.columns:
        if dict(df.dtypes)[c]=="string":
            df=df.withColumn(c,when(col(c)=="-1",None).otherwise(col(c)))
        else:
            df=df.withColumn(c,when(col(c)==-1,None).otherwise(col(c)))
    return df

In [13]:
df=clean_dataset(df)

In [14]:
df.show()

+--------------+----------+---------+---------+--------------+-------+-------------+
|          name|      diet|prep_time|cook_time|flavor_profile| course|        state|
+--------------+----------+---------+---------+--------------+-------+-------------+
|    Balu shahi|vegetarian|       45|       25|         sweet|dessert|  West Bengal|
|        Boondi|vegetarian|       80|       30|         sweet|dessert|    Rajasthan|
|Gajar ka halwa|vegetarian|       15|       60|         sweet|dessert|       Punjab|
|        Ghevar|vegetarian|       15|       30|         sweet|dessert|    Rajasthan|
|   Gulab jamun|vegetarian|       15|       40|         sweet|dessert|  West Bengal|
|        Imarti|vegetarian|       10|       50|         sweet|dessert|  West Bengal|
|        Jalebi|vegetarian|       10|       50|         sweet|dessert|Uttar Pradesh|
|    Kaju katli|vegetarian|       10|       20|         sweet|dessert|         NULL|
|      Kalakand|vegetarian|       20|       30|         sweet|des

In [15]:
#1.Find out how many unique dishes are present.
df.select('name').distinct().show(truncate=False)

+-----------------+
|name             |
+-----------------+
|Kheer            |
|Poriyal          |
|Sabudana Khichadi|
|Gajar ka halwa   |
|Keerai sadam     |
|Dalithoy         |
|Mihidana         |
|Singori          |
|Uttapam          |
|Gavvalu          |
|Kakinada khaja   |
|Chak Hao Kheer   |
|Pachadi          |
|Vindaloo         |
|Gheela Pitha     |
|Idiappam         |
|Keerai kootu     |
|Idli             |
|Saath            |
|Rabri            |
+-----------------+
only showing top 20 rows



In [16]:
#2. Which state has more dishes? 
df.groupBy('state').count().orderBy('count', ascending=False).limit(1).show()

+-------+-----+
|  state|count|
+-------+-----+
|Gujarat|   35|
+-------+-----+



In [20]:
#3. How many dishes from state Karnataka? 
df.select('state').where(col('state')=='Karnataka').count()

6

In [21]:
#4. List number of unique regions
df.filter(col("state").isNotNull()).select("state").distinct().count()

24

In [22]:
df.filter(col("state").isNotNull()).select("state").distinct().show()

+---------------+
|          state|
+---------------+
|       Nagaland|
|      Karnataka|
|         Odisha|
|         Kerala|
|     Tamil Nadu|
|   Chhattisgarh|
| Andhra Pradesh|
| Madhya Pradesh|
|         Punjab|
|        Manipur|
|Jammu & Kashmir|
|            Goa|
|        Haryana|
|        Gujarat|
|      Rajasthan|
|          Assam|
|   NCT of Delhi|
|    West Bengal|
|    Maharashtra|
|      Telangana|
+---------------+
only showing top 20 rows



In [24]:
#6. List unique 'flavor_profile' and 'course' 
df.select("flavor_profile","course").distinct().show()

+--------------+-----------+
|flavor_profile|     course|
+--------------+-----------+
|        bitter|      snack|
|         spicy|    starter|
|          sour|main course|
|         sweet|main course|
|        bitter|main course|
|         spicy|      snack|
|         sweet|    dessert|
|         spicy|main course|
|          NULL|      snack|
|          NULL|main course|
+--------------+-----------+



In [27]:
#7. Which state has more 'main course'?
df.select("course","state").where(col("course")=="main course").groupBy("state").count().orderBy('count',ascending=False).limit(1).show()

+------+-----+
| state|count|
+------+-----+
|Punjab|   28|
+------+-----+



In [31]:
#8. Give the %of dishes from each region. 
total_dishes=df.count()
df.groupBy("state").agg(count("*").alias("dish_count")).withColumn("percentage",round((col("dish_count")/total_dishes)*100,2)).orderBy("percentage",ascending=False).show()

+---------------+----------+----------+
|          state|dish_count|percentage|
+---------------+----------+----------+
|        Gujarat|        35|     13.73|
|         Punjab|        32|     12.55|
|    Maharashtra|        30|     11.76|
|           NULL|        24|      9.41|
|    West Bengal|        24|      9.41|
|          Assam|        21|      8.24|
|     Tamil Nadu|        20|      7.84|
| Andhra Pradesh|        10|      3.92|
|  Uttar Pradesh|         9|      3.53|
|         Kerala|         8|      3.14|
|         Odisha|         7|      2.75|
|      Karnataka|         6|      2.35|
|      Rajasthan|         6|      2.35|
|      Telangana|         5|      1.96|
|            Goa|         3|      1.18|
|          Bihar|         3|      1.18|
| Madhya Pradesh|         2|      0.78|
|        Manipur|         2|      0.78|
|Jammu & Kashmir|         2|      0.78|
|       Nagaland|         1|      0.39|
+---------------+----------+----------+
only showing top 20 rows



In [34]:
#9. List the states which has more dishes from each region. 
df.groupBy("state").agg(count("*").alias("dish_count")).orderBy("dish_count",ascending=False).show()

+---------------+----------+
|          state|dish_count|
+---------------+----------+
|        Gujarat|        35|
|         Punjab|        32|
|    Maharashtra|        30|
|           NULL|        24|
|    West Bengal|        24|
|          Assam|        21|
|     Tamil Nadu|        20|
| Andhra Pradesh|        10|
|  Uttar Pradesh|         9|
|         Kerala|         8|
|         Odisha|         7|
|      Karnataka|         6|
|      Rajasthan|         6|
|      Telangana|         5|
|            Goa|         3|
|          Bihar|         3|
| Madhya Pradesh|         2|
|        Manipur|         2|
|Jammu & Kashmir|         2|
|       Nagaland|         1|
+---------------+----------+
only showing top 20 rows

